In [18]:
import sys
import os

# SPARK_HOME path
os.environ["SPARK_HOME"] = "/home/fragkiska/spark"

# Add pyspark to Python path
sys.path.append("/home/fragkiska/spark/python")
sys.path.append("/home/fragkiska/spark/python/lib/py4j-0.10.9.7-src.zip")


import pyspark
from pyspark.sql import SparkSession

In [19]:
from pyspark.sql import SparkSession
from pyspark.sql.functions import *

spark = (
    SparkSession.builder
    .appName("Query2-AdvancedDBs")
    .config("spark.executor.instances", "4")
    .config("spark.executor.cores", "1")
    .config("spark.executor.memory", "2g")
    .getOrCreate()
)


In [20]:
from pyspark.sql.functions import *


from pathlib import Path

project_root = Path.cwd()

data_dir = project_root / "data"

crime_data_2010_2019 = data_dir / "LA_Crime_Data_2010_2019.csv"
crime_data_2020_2025 = data_dir / "LA_Crime_Data_2020_2025.csv"
mo_path = str(data_dir / "MO_codes.txt")

mo_df = (
    spark.read.text(mo_path)
        .withColumn("code", split(col("value"), " ")[0])
        .withColumn("description", expr("substring(value, length(code) + 2)"))
        .select("code", "description")
)

df1 = spark.read.csv(str(crime_data_2010_2019), header=True, inferSchema=True)
df2 = spark.read.csv(str(crime_data_2020_2025), header=True, inferSchema=True)

crime_data = df1.unionByName(df2)

crime_mo = (
    crime_data
    .filter(col("Mocodes").isNotNull())
    .withColumn("mo_code", explode(split(col("Mocodes"), " ")))
    .filter(col("mo_code") != "")                 # remove empty tokens
)

crime_mo.show(5)   


+--------+--------------------+--------------------+--------+----+---------+-----------+--------+------+--------------------+--------------+--------+--------+------------+---------+--------------------+--------------+-----------+------+------------+--------+--------+--------+--------+--------------------+--------------------+-------+---------+-------+
|   DR_NO|           Date Rptd|            DATE OCC|TIME OCC|AREA|AREA NAME|Rpt Dist No|Part 1-2|Crm Cd|         Crm Cd Desc|       Mocodes|Vict Age|Vict Sex|Vict Descent|Premis Cd|         Premis Desc|Weapon Used Cd|Weapon Desc|Status| Status Desc|Crm Cd 1|Crm Cd 2|Crm Cd 3|Crm Cd 4|            LOCATION|        Cross Street|    LAT|      LON|mo_code|
+--------+--------------------+--------------------+--------+----+---------+-----------+--------+------+--------------------+--------------+--------+--------+------------+---------+--------------------+--------------+-----------+------+------------+--------+--------+--------+--------+-------

In [21]:
# Dataframe implementation

import time

start_df = time.time()

mo_counts = (
    crime_mo.groupBy("mo_code")
            .count()
            .orderBy(col("count").desc())
)


mo_joined = (
    mo_counts.join(mo_df, mo_counts.mo_code == mo_df.code, "left")
             .select("mo_code", "description", "count")
             .orderBy(col("count").desc())
)

print("=== DEFAULT JOIN STRATEGY ===")
mo_joined.explain(False)


mo_joined.show(5, truncate=False)

end_df = time.time()
print(f"DataFrame Implementation Time: {end_df - start_df:.2f} sec")

=== DEFAULT JOIN STRATEGY ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#1463L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(count#1463L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=2881]
      +- Project [mo_code#1230, description#1048, count#1463L]
         +- BroadcastHashJoin [mo_code#1230], [code#1044], LeftOuter, BuildRight, false
            :- HashAggregate(keys=[mo_code#1230], functions=[count(1)])
            :  +- Exchange hashpartitioning(mo_code#1230, 200), ENSURE_REQUIREMENTS, [plan_id=2874]
            :     +- HashAggregate(keys=[mo_code#1230], functions=[partial_count(1)])
            :        +- Filter NOT (mo_code#1230 = )
            :           +- Generate explode(split(Mocodes#1081,  , -1)), false, [mo_code#1230]
            :              +- Union
            :                 :- Filter isnotnull(Mocodes#1081)
            :                 :  +- FileScan csv [Mocodes#1081] Batched: false, DataFilters: [isnotnull(Mo

+-------+---------------------+-------+
|mo_code|description          |count  |
+-------+---------------------+-------+
|0344   |Removes vict property|1002900|
|1822   |Stranger             |548422 |
|0416   |Hit-Hit w/ weapon    |404773 |
|0329   |Vandalized           |377536 |
|0913   |Victim knew Suspect  |278618 |
+-------+---------------------+-------+
only showing top 5 rows

DataFrame Implementation Time: 9.03 sec


In [22]:
# Dataframe implementation

mo_counts = (
    crime_mo.groupBy("mo_code")
            .count()
            .orderBy(col("count").desc())
)

mo_joined = (
    mo_counts.join(mo_df, mo_counts.mo_code == mo_df.code, "left")
             .select("mo_code", "description", "count")
             .orderBy(col("count").desc())
)

print("=== DEFAULT JOIN STRATEGY ===")
mo_joined.explain(False)

print("\n=== BROADCAST JOIN ===")
mo_broadcast = (
    mo_counts.hint("broadcast")
             .join(mo_df, mo_counts.mo_code == mo_df.code, "left")
             .select("mo_code", "description", "count")
)
mo_broadcast.explain(False)

print("\n=== SHUFFLE_HASH JOIN ===")
mo_hash = (
    mo_counts.hint("shuffle_hash")
             .join(mo_df, mo_counts.mo_code == mo_df.code, "left")
             .select("mo_code", "description", "count")
)
mo_hash.explain(False)

print("\n=== MERGE JOIN ===")
mo_merge = (
    mo_counts.hint("merge")
             .join(mo_df, mo_counts.mo_code == mo_df.code, "left")
             .select("mo_code", "description", "count")
)
mo_merge.explain(False)

print("\n=== SHUFFLE_REPLICATE_NL JOIN ===")
mo_nl = (
    mo_counts.hint("shuffle_replicate_nl")
             .join(mo_df, mo_counts.mo_code == mo_df.code, "left")
             .select("mo_code", "description", "count")
)
mo_nl.explain(False)

mo_joined.show(5, truncate=False)



=== DEFAULT JOIN STRATEGY ===
== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Sort [count#1525L DESC NULLS LAST], true, 0
   +- Exchange rangepartitioning(count#1525L DESC NULLS LAST, 200), ENSURE_REQUIREMENTS, [plan_id=3140]
      +- Project [mo_code#1230, description#1048, count#1525L]
         +- BroadcastHashJoin [mo_code#1230], [code#1044], LeftOuter, BuildRight, false
            :- HashAggregate(keys=[mo_code#1230], functions=[count(1)])
            :  +- Exchange hashpartitioning(mo_code#1230, 200), ENSURE_REQUIREMENTS, [plan_id=3133]
            :     +- HashAggregate(keys=[mo_code#1230], functions=[partial_count(1)])
            :        +- Filter NOT (mo_code#1230 = )
            :           +- Generate explode(split(Mocodes#1081,  , -1)), false, [mo_code#1230]
            :              +- Union
            :                 :- Filter isnotnull(Mocodes#1081)
            :                 :  +- FileScan csv [Mocodes#1081] Batched: false, DataFilters: [isnotnull(Mo

25/12/06 21:26:37 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


== Physical Plan ==
AdaptiveSparkPlan isFinalPlan=false
+- Project [mo_code#1230, description#1048, count#1525L]
   +- SortMergeJoin [mo_code#1230], [code#1044], LeftOuter
      :- Sort [mo_code#1230 ASC NULLS FIRST], false, 0
      :  +- HashAggregate(keys=[mo_code#1230], functions=[count(1)])
      :     +- Exchange hashpartitioning(mo_code#1230, 200), ENSURE_REQUIREMENTS, [plan_id=3341]
      :        +- HashAggregate(keys=[mo_code#1230], functions=[partial_count(1)])
      :           +- Filter NOT (mo_code#1230 = )
      :              +- Generate explode(split(Mocodes#1081,  , -1)), false, [mo_code#1230]
      :                 +- Union
      :                    :- Filter isnotnull(Mocodes#1081)
      :                    :  +- FileScan csv [Mocodes#1081] Batched: false, DataFilters: [isnotnull(Mocodes#1081)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/mnt/c/Users/user/Desktop/advanced_dbs/Advanced-Databases-Project..., PartitionFilters: [], PushedFilters: [IsNotNul

+-------+---------------------+-------+
|mo_code|description          |count  |
+-------+---------------------+-------+
|0344   |Removes vict property|1002900|
|1822   |Stranger             |548422 |
|0416   |Hit-Hit w/ weapon    |404773 |
|0329   |Vandalized           |377536 |
|0913   |Victim knew Suspect  |278618 |
+-------+---------------------+-------+
only showing top 5 rows



In [23]:
import time

def timed_run(df, label):
    start = time.time()
    df.count()   # ACTION → triggers execution
    end = time.time()
    print(f"{label}: {end - start:.3f} sec")


timed_run(mo_joined, "DEFAULT")
timed_run(mo_broadcast, "BROADCAST")
timed_run(mo_hash, "SHUFFLE_HASH")
timed_run(mo_merge, "MERGE")
timed_run(mo_nl, "SHUFFLE_REPLICATE_NL")


25/12/06 21:26:55 WARN HintErrorLogger: Hint (strategy=broadcast) is not supported in the query: build left for left outer join.


DEFAULT: 9.417 sec


BROADCAST: 7.871 sec


SHUFFLE_HASH: 9.361 sec


MERGE: 7.594 sec


SHUFFLE_REPLICATE_NL: 7.637 sec


In [24]:
# RDD version
start_rdd = time.time()
rdd = (
    crime_data
    .filter(col("Mocodes").isNotNull())
    .select("Mocodes")
    .rdd
    .flatMap(lambda row: row["Mocodes"].split(" "))
    .filter(lambda x: x != "")
    .map(lambda code: (code, 1))
    .reduceByKey(lambda x, y: x + y)
)

mo_counts_rdd = rdd.toDF(["mo_code", "count"])

result_rdd = (
    mo_counts_rdd.join(mo_df, mo_counts_rdd.mo_code == mo_df.code, "left")
                 .select("mo_code", "description", "count")
                 .orderBy(col("count").desc())
)

result_rdd.show(5, truncate=False)

end_rdd = time.time()
print(f"RDD Implementation Time: {end_rdd - start_rdd:.2f} sec")

+-------+---------------------+-------+
|mo_code|description          |count  |
+-------+---------------------+-------+
|0344   |Removes vict property|1002900|
|1822   |Stranger             |548422 |
|0416   |Hit-Hit w/ weapon    |404773 |
|0329   |Vandalized           |377536 |
|0913   |Victim knew Suspect  |278618 |
+-------+---------------------+-------+
only showing top 5 rows

RDD Implementation Time: 27.09 sec
